In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [3]:
df.shape

(6000, 785)

In [4]:
x = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [5]:
xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size=0.3)

In [6]:
xTrain = xTrain/255.00
xTest = xTest/255.00

In [7]:
xTrain

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.00784314, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(4200, 784))

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

In [9]:
class CustomData(Dataset):

    def __init__(self, features, label):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.label = torch.tensor(label, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.label[index]

In [10]:
trainData = CustomData(xTrain, yTrain)

In [11]:
testData = CustomData(xTest, yTest)

In [12]:
trainDataLoader = DataLoader(trainData, batch_size=64, shuffle=True)
testDataLoader = DataLoader(testData, batch_size=64, shuffle=False)

In [13]:
import torch.nn as nn

In [62]:
class NeuralNetwork(nn.Module):

    def __init__(self, feature_len):
        super().__init__()

        self.linear = nn.Sequential(
            nn.Linear(feature_len, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(32, 10)
        )

    def forward(self, features):
        predict = self.linear(features)
        return predict

In [63]:
epochs = 50
learning_rate = 0.01

In [64]:
model = NeuralNetwork(xTrain.shape[1])
loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-2)

In [65]:
for epoch in range(epochs):
    for feature, label in trainDataLoader:
        y_pred = model(feature)
        loss = loss_fn(y_pred, label)
        optim.zero_grad()
        loss.backward()
        optim.step()
    print(f"Epochs: {epoch + 1}, Loss: {loss.item()}")

Epochs: 1, Loss: 1.516684889793396
Epochs: 2, Loss: 1.3281991481781006
Epochs: 3, Loss: 1.1306028366088867
Epochs: 4, Loss: 0.9925691485404968
Epochs: 5, Loss: 0.8868996500968933
Epochs: 6, Loss: 0.887877345085144
Epochs: 7, Loss: 0.8731958270072937
Epochs: 8, Loss: 0.9268960952758789
Epochs: 9, Loss: 0.6510778069496155
Epochs: 10, Loss: 0.6255131959915161
Epochs: 11, Loss: 0.8659882545471191
Epochs: 12, Loss: 0.7176074385643005
Epochs: 13, Loss: 0.7227453589439392
Epochs: 14, Loss: 0.54820716381073
Epochs: 15, Loss: 0.692581057548523
Epochs: 16, Loss: 0.5318070650100708
Epochs: 17, Loss: 0.5146711468696594
Epochs: 18, Loss: 0.6479064226150513
Epochs: 19, Loss: 0.4909742474555969
Epochs: 20, Loss: 0.6083791255950928
Epochs: 21, Loss: 0.561211347579956
Epochs: 22, Loss: 0.599013090133667
Epochs: 23, Loss: 0.4957549571990967
Epochs: 24, Loss: 0.4087652266025543
Epochs: 25, Loss: 0.5022261142730713
Epochs: 26, Loss: 0.4218953549861908
Epochs: 27, Loss: 0.4426921010017395
Epochs: 28, Loss:

In [66]:
model.eval()

NeuralNetwork(
  (linear): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=False)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=32, bias=False)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [67]:
x = torch.randn(3, 784)

In [68]:
x.shape

torch.Size([3, 784])

In [69]:
with torch.no_grad():
    output = model(x)

In [70]:
output

tensor([[-2.8325,  2.8446, -2.1233, -1.5344, -2.5954,  8.1150, -0.1333, -2.3645,
          2.8823,  1.0163],
        [-2.7660, -1.0448, -1.4450, -0.8497, -4.0008, 11.5127, -0.4712, -3.0012,
         -0.2862,  2.9589],
        [-4.0531, -4.3949, -4.1612,  2.2492,  3.1584,  9.3665,  0.1037, -2.5601,
          2.9574, -1.3615]])

In [71]:
torch.max(output, 1)

torch.return_types.max(
values=tensor([ 8.1150, 11.5127,  9.3665]),
indices=tensor([5, 5, 5]))

In [72]:
total = 0
correct = 0

with torch.no_grad():
    for feature, label in testDataLoader:
        model_output = model(feature)
        _, prediction = torch.max(model_output, 1)
        total += label.shape[0]
        correct += (prediction == label).sum().item()
print(correct/total)

0.8411111111111111


In [73]:
total = 0
correct = 0

with torch.no_grad():
    for feature, label in trainDataLoader:
        model_output = model(feature)
        _, prediction = torch.max(model_output, 1)
        total += label.shape[0]
        correct += (prediction == label).sum().item()
print(correct/total)

0.9776190476190476
